# PubMed pathogen category assignment

This notebook reads the accepted abstracts from notebook 02, classifies
abstract-level animal-infection and zoonosis evidence, and derives the
corpus-relative categories 1, 2, and 3. It writes a pathogen-level category
table under outputs/pubmed_screening/.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'assets').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'assets').exists():
    raise FileNotFoundError('Could not locate the repository assets directory.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from graphicalizer import (
    OpenAIChatCompleter,
    derive_category_counts,
    load_corpus_articles,
    screen_pubmed_corpus,
)

## Load configuration for this notebook

In [ ]:
from graphicalizer.notebook_config import (
    configured_env,
    debug_pathogens,
    limit_debug_abstracts,
    load_notebook_config,
    resolve_config_path,
)

CONFIG = load_notebook_config(PROJECT_ROOT)
COMMON = CONFIG['common']
SETTINGS = CONFIG['notebook_03_pubmed_pathogen_category']
DEBUG_MODE = COMMON['debug_mode']
DEBUG_PATHOGENS = debug_pathogens(COMMON)

OUTPUT_DIR = resolve_config_path(
    PROJECT_ROOT,
    Path(COMMON['output_root'])
    / COMMON['corpus_output_subdir']
    / ('debug' if DEBUG_MODE else ''),
)
REFINED_CORPUS_PATH = OUTPUT_DIR / 'refined_corpus_articles.parquet'
CORPUS_PATHOGENS = (
    list(DEBUG_PATHOGENS)
    if DEBUG_MODE
    else COMMON['corpus_pathogens']
)
CORPUS_START_YEAR = COMMON['corpus_start_year']
CORPUS_END_YEAR = COMMON['corpus_end_year']
PATHOGENS = DEBUG_PATHOGENS if DEBUG_MODE else COMMON['pathogens']
OPENAI_MODEL = configured_env(COMMON, 'openai_model_env', COMMON['openai_model_default'])
OPENAI_API_KEY = configured_env(COMMON, 'openai_api_key_env')
LLM_MAX_TOKENS = SETTINGS['openai_max_tokens']
LLM_RETRIES = COMMON['llm_retries']
LLM_MAX_CALLS = COMMON['llm_max_calls']
LLM_SAVE_EVERY = COMMON['llm_save_every']
RESUME = COMMON['resume']
RETRY_FAILED_LLM_ROWS = COMMON['retry_failed_llm_rows']


## Load and filter the accepted corpus

In [ ]:
corpus = load_corpus_articles(
    REFINED_CORPUS_PATH,
    pathogens=CORPUS_PATHOGENS,
    start_year=CORPUS_START_YEAR,
    end_year=CORPUS_END_YEAR,
)
corpus = limit_debug_abstracts(corpus, COMMON)
print('Accepted corpus rows:', len(corpus))
if corpus.empty:
    raise ValueError('No accepted abstracts remain after category-stage filtering.')
display(corpus.groupby('pathogen').size().rename('abstracts').reset_index())

## Classify animal-infection and zoonosis evidence

In [ ]:
if not OPENAI_API_KEY:
    raise RuntimeError('Set OPENAI_API_KEY before running the category screen.')
llm = OpenAIChatCompleter(OPENAI_MODEL)
screening_run = screen_pubmed_corpus(
    corpus,
    PATHOGENS,
    llm,
    OUTPUT_DIR,
    model=OPENAI_MODEL,
    max_tokens=LLM_MAX_TOKENS,
    retries=LLM_RETRIES,
    max_llm_calls=LLM_MAX_CALLS,
    save_every=LLM_SAVE_EVERY,
    resume=RESUME,
    retry_failed=RETRY_FAILED_LLM_ROWS,
)
print('Category decisions:', len(screening_run.screening))
print('Classification failures:', len(screening_run.failures))

## Derive categories and associate them with pathogens

In [ ]:
screened_with_categories, zoonosis_timeline, category_counts = derive_category_counts(
    screening_run.screening,
    OUTPUT_DIR,
)
pathogen_category_table = pd.read_parquet(OUTPUT_DIR / 'pathogen_category_table.parquet')
print('First confirmed zoonosis by pathogen:')
display(zoonosis_timeline)
print('Pathogen category table:')
display(pathogen_category_table)
print('Yearly and total category counts:')
display(category_counts.sort_values(['pathogen', 'publication_year', 'category'], na_position='first'))

Category 1 means no confirmed zoonosis evidence was found in the searched
corpus for that pathogen; it is not proof that the pathogen has never
undergone zoonosis. Review-required and failed rows are excluded from final
counts. The category table is a corpus-level summary, not a biological truth
label.

In [ ]:
print('Output directory:', OUTPUT_DIR)
print('Files:', sorted(path.name for path in OUTPUT_DIR.glob('*') if path.is_file()))
print('Review-required rows:', int(screened_with_categories['review_required'].fillna(True).sum()))
display(screened_with_categories[[
    'pathogen', 'pmid', 'llm_category', 'final_category',
    'target_pathogen_supported', 'animal_infection_supported',
    'zoonosis_supported', 'confidence', 'review_required', 'rationale'
]].head(20))